# MouseFull training and evaluation

Joint **D1 + D3** training with eleven separate anatomical classes and the existing SHAM/TBI classifier. This notebook uses the same fixed-split runner, checkpoint selection, recovery and full-volume evaluation as the TBI mouse workflow. Use the **rnet** environment and inspect data contracts before enabling training.

The label order matches the old `DA_3DRNET_Training.ipynb`. Model settings below explicitly use the current original-width 3DR-Net configuration. Builder defaults, loss weights and dropout behavior are unchanged; the corrected ROI objective is selected explicitly.

In [1]:
from pathlib import Path
import sys

REPO_ROOT: Path = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/experiments/config.py").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.experiments.training import configure_runtime

TRAINING_SEED: int = 42
runtime = configure_runtime(seed=TRAINING_SEED, memory_growth=True, mixed_precision=False)
runtime
import nibabel as nib

REFERENCE: Path = (
    REPO_ROOT / "example" / "RARE" / "TBI_fm_19_49"
    / "Anat" / "TBI_fm_19_49_N4.nii.gz"
)
ref_img = nib.load(REFERENCE)
reference_axes = nib.aff2axcodes(ref_img.affine)
print("Reference axes:", reference_axes, "(left-right is the R/L axis)")


{'tensorflow': '2.10.0',
 'gpu_devices': ['/physical_device:GPU:0'],
 'precision': 'float32',
 'seed': 42}

## Complete MouseFull configuration

Classes are not merged: contra/ipsilateral corpus callosum, ventricles, hippocampus and cortex remain separate. Extra anatomy outside the selected mapping is projected to task background. D1 scans with only the four target regions have unavailable extended anatomy. Verified lesion-free D3 SHAM scans always use available empty lesion targets, as in the original pipeline. The generator marks unavailable channels with **-1** and the corrected loss consumes logits. Inference exports full softmax probabilities.

Unannotated channels must not be interpreted as negative examples. The corrected loss retains its documented positive-available-target policy; this does not establish complete background annotation. Review the declared dataset/background annotation policy before manuscript training.

`animal_ids` can group confirmed duplicate biological animals across cohorts. IDs are never guessed or merged from similar filenames. D2 and external D4 are excluded. TBI backbone initialization is configured explicitly below.

Full-volume outputs use `original_label_ids=True`. `postprocessing_settings` controls brain-mask cleanup and ventricle separation. Thresholds are voxel counts on the 0.1 mm grid; categorical ROI masks are not cleaned as binary masks. Merged D1/rat ventricles are separated before mapping to original IDs 3 (contra), 13 (ipsi), and 21 (third ventricle). MouseFull keeps its directly predicted sides. GT uses the selected original annotations, never the prediction postprocessor.

The historical `default_hemisphere="right"` selects the plane side containing array origin, rather than deriving anatomical right from the affine. It is used when no sufficiently large lesion is predicted. Splitting requires at least three predicted third-ventricle voxels; missing midline predictions raise an explicit scan-specific error.


`USE_TBI_PRETRAINING=True` prepares feature transfer from a previous TBI mouse run. Set `TBI_RUN` to its run folder and `TBI_FOLD` to its source fold. The loader reads `experiment.json` and selects the committed best-weight snapshot, so a completed epoch is sufficient even if training was interrupted. It reconstructs the four-class source graph and verifies backbone topology and weight shapes before transferring encoder/decoder features. The eleven-class ROI head, brain head and SHAM/TBI classifier start fresh; channels with different label meanings are not copied. With `FREEZE_BACKBONE=False`, all transferred features are fine-tuned. Architecture, loss and learning rate keep their existing settings.

No checkpoint is chosen automatically. With `TBI_RUN=None`, data previews work but training requires a source run. Set `USE_TBI_PRETRAINING=False` to deliberately train from scratch. Initialization details and source hashes are saved in each fold's `initialization.json`. `RESUME_RUN` restores the MouseFull fitting state rather than restarting from TBI weights.

TBI pretraining must exclude the animals used for MouseFull evaluation, including validation animals if they are intended as unseen checkpoint-selection data. MouseFull contains D1, so a general D1-trained checkpoint can overlap later MouseFull splits. For single training, align the target validation/test selection with animals unseen in source training and source selection-validation. For K-Fold, one shared checkpoint is valid only if it excludes every evaluated animal across those folds; otherwise use separately trained sources matching each fold's exclusions. This notebook's shared `TBI_RUN` is an explicit common initialization and does not automatically make a source split independent of the target folds.


In [ ]:
from src.experiments.config import (
    DataConfig, DatasetSpec, EvaluationPatchConfig, ExperimentConfig, FixedSplitConfig, InferenceConfig, InitializationConfig,
    LabelConfig, LossConfig, ModelConfig, PostprocessingConfig, PreprocessingConfig,
)
from src.evaluation.k_fold import TrainingConfig

DATA_ROOT: Path = Path("D:/Other files/TESI_CNN/MANUSCRIPT_DATASET/normalized")
RUN_TRAINING: bool = False
USE_TBI_PRETRAINING: bool = True
TBI_RUN: Path | None = None  # Previous notebook 01/02 run folder, including interrupted runs.
TBI_FOLD: int = 0  # Source fold; single-run TBI training uses fold 0.
FREEZE_BACKBONE: bool = False  # False fine-tunes all transferred feature layers.
RESUME_RUN: Path | None = None
SPLIT_FILE: Path | None = None

training = TrainingConfig(
    model_name="mouse_full_3drnet", epochs=300, batch_size=8,
    seed=TRAINING_SEED, data_multiplier=3, output_dir=str(REPO_ROOT / "results"),
    monitor="val_loss", monitor_mode="min", default_callbacks=False,
    fit_kwargs={"verbose": 1},
)
datasets = (
    DatasetSpec(name="D1", annotation_scope="full_volume"),
    # All D3 SHAM scans are verified lesion-free: supervise an empty lesion mask.
    DatasetSpec(name="D3", annotation_scope="full_volume"),
)
animal_ids: dict[str, str] = {}
# Fractions of all animals; saved split files keep their exact memberships.
split_settings = FixedSplitConfig(
    validation_fraction=0.2, test_fraction=0.1,
)

data_settings = DataConfig(root=DATA_ROOT, datasets=datasets, animal_ids=animal_ids)
preprocessing = PreprocessingConfig(
    reference=REFERENCE, patch_size=(80, 80, 80), spacing_mm=(0.1, 0.1, 0.1),
    correct_units=True, augment=True, random_crop=True, crop_mode="center",
    # Native-grid MRI degradation; bounds retain their earlier magnitudes.
    intensity_augmentation_grid="native",
    blur_sigma_range=(0.0, 0.6), blur_sigma_units="voxels",
    noise_std_range=(0.0, 0.05), noise_std_units="intensity",
    flip_probability=0.0,
)

# Preserve the historical MouseFull channel order; keep each side separate.
label_mapping: dict[int, int] = {
    0: 0, 1: 1, 2: 2, 3: 3, 5: 4, 6: 5,
    12: 6, 13: 7, 15: 8, 16: 9, 21: 10,
}
class_names: tuple[str, ...] = (
    "Background", "Lesion", "Contra corpus callosum", "Contra ventricle",
    "Contra hippocampus", "Contra cortex", "Ipsi corpus callosum",
    "Ipsi ventricle", "Ipsi hippocampus", "Ipsi cortex", "Third ventricle",
)
labels = LabelConfig(mapping=label_mapping, names=class_names, unknown="background")
# Sample once; validation/test reuse evaluation_patches.json across epochs and runs.
evaluation_patch_settings = EvaluationPatchConfig(
    patches_per_scan=3, sampling="center_biased", seed=42,
    left_right_pairs=False,  # D1/rats: optional paired LR views; MouseFull: keep False.
)

model_settings = ModelConfig(
    architecture="3drnet", filters=(16, 32, 64, 128, 256),
    residual=True, attention=True, max_pooling=True, classifier=True,
)
loss_settings = LossConfig(
    regions="dice_ce_gradient_fix", gradient_fix_version=2, gamma=0.8, brain_alpha=0.6, batch_wise=True,
    # Total loss = weighted sum of the active output-head losses.
    regions_weight=1.0, brain_weight=1.0, classifier_weight=1.0,
)
# Full volumes: postprocess first, then restore original annotation IDs.
postprocessing_settings = PostprocessingConfig(
    clean_brain_mask=True,
    fill_small_holes=True, holes_max_area=20000,
    remove_small_objects=True, object_min_area=30000,
    split_ventricles=False, use_centroids=False,
    default_hemisphere="right", lesion_voxels_threshold=10,
)
inference_settings = InferenceConfig(
    enabled=True, stride=40, save_predictions=True,
    original_label_ids=True, postprocessing=postprocessing_settings,
)
# Transfer feature extraction/decoding, keeping all MouseFull output heads fresh.
initialization = InitializationConfig()
if USE_TBI_PRETRAINING:
    if TBI_RUN is not None:
        initialization = InitializationConfig.from_run(
            run_dir=TBI_RUN, fold=TBI_FOLD, freeze_backbone=FREEZE_BACKBONE,
        )
        print("TBI initialization weights:", initialization.checkpoint)
    else:
        print("Data preview only: set TBI_RUN before training, or disable USE_TBI_PRETRAINING for scratch.")

config = ExperimentConfig(
    data=data_settings, preprocessing=preprocessing, training=training,
    evaluation_patches=evaluation_patch_settings,
    fixed_split=split_settings,
    model=model_settings, labels=labels, loss=loss_settings,
    inference=inference_settings, initialization=initialization,
    learning_rate=0.001, split_seed=42,
)
config.validate()

## Configure callbacks directly

Fixed-split training uses the original LR reduction, TensorBoard histograms, early stopping and CSV logging. Edit the callback constructors or the list in the next cell; the training call explicitly receives `callbacks=callbacks`. No callback is added by a hidden experiment factory. `TrainingConfig.default_callbacks=False` prevents duplicate automatic logging/LR callbacks. Checkpoint selection uses the visible `TrainingConfig.monitor` and `monitor_mode` settings.

Relative log paths resolve inside each run/fold, so folds and study variants do not share files. The runner copies unused callback instances for every fit and saves their settings in `callbacks.json`; resume requires the same saved settings. CSV logs append when resuming.

The runner always saves best weights and epoch recovery state, even when this list is empty. It restores the selected best checkpoint before evaluation. For that reason early stopping uses `restore_best_weights=False`: restoring weights inside its epoch-end hook would mix earlier weights with the latest optimizer in the recovery checkpoint. The final best-weight behavior matches the original workflow.


In [ ]:
import tensorflow as tf

# Relative logging paths are resolved inside this run's fold directory.
log_dir: str = "tensorboard"
csv_filename: str = "training.csv"
csv_logger = tf.keras.callbacks.CSVLogger(filename=csv_filename, append=True)

callbacks: list[tf.keras.callbacks.Callback] = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", mode="min", factor=0.5, patience=10,
        min_lr=1e-7, verbose=1,
    ),
    tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", mode="min", patience=20, verbose=1,
        restore_best_weights=False,  # The runner reloads the best saved weights before evaluation.
    ),
    csv_logger,
]


## Splits and declared annotation coverage

Load a chosen `SPLIT_FILE` for manuscript work and matched comparisons, or inspect the deterministic unstratified split below. Every registered animal belongs to exactly one role. `RESUME_RUN` reloads and validates its saved split. Coverage comes from annotation metadata, not the presence of a structure in the mask. Review the TBI/SHAM and modality balance in the manifest; no stratification is implied.

In [3]:
from IPython.display import display
from src.experiments.data import SubjectSplit, prepare_data

prepared = prepare_data(config)

if RESUME_RUN is not None:
    saved_split = RESUME_RUN / "splits.json"
    split = SubjectSplit.load(saved_split, prepared.subjects)
    if SPLIT_FILE is not None:
        requested_split = SubjectSplit.load(SPLIT_FILE, prepared.subjects)
        if requested_split != split:
            raise ValueError("SPLIT_FILE differs from the resumed run's split")
elif SPLIT_FILE is not None:
    split = SubjectSplit.load(SPLIT_FILE, prepared.subjects)
else:
    split = prepared.make_split()

display(prepared.split_summary(split))
manifest = prepared.subjects.manifest(prepared.subjects.subject_ids)
coverage = prepared.annotation_summary()
display(coverage)
columns = ["Subject", "Dataset", "OriginalID", "Timepoint", "Scan"]
display(manifest[columns])

,Role,Animals,Scans
0,train,138,138
1,validation,40,40
2,test,20,20


,Dataset,ClassIndex,Class,SourceLabels,AnnotatedScans,UnannotatedScans,TotalScans
0,D1,0,Background,[0],112,0,112
1,D1,1,Lesion,[1],112,0,112
2,D1,2,Contra corpus callosum,[2],9,103,112
3,D1,3,Contra ventricle,[3],112,0,112
4,D1,4,Contra hippocampus,[5],9,103,112
5,D1,5,Contra cortex,[6],9,103,112
6,D1,6,Ipsi corpus callosum,[12],9,103,112
7,D1,7,Ipsi ventricle,[13],112,0,112
8,D1,8,Ipsi hippocampus,[15],9,103,112
9,D1,9,Ipsi cortex,[16],9,103,112


,Subject,Dataset,OriginalID,Timepoint,Scan
0,D1/SUBJ_0001,D1,Isch_PAL_PPIAKO_1767,None,SUBJ_0001
1,D1/SUBJ_0002,D1,Isch_PAL_PPIAKO_1777,None,SUBJ_0002
2,D1/SUBJ_0003,D1,Isch_PAL_PPIAKO_1781,None,SUBJ_0003
3,D1/SUBJ_0004,D1,Isch_PAL_PPIAKO_1782,None,SUBJ_0004
4,D1/SUBJ_0005,D1,Isch_PAL_PPIAKO_1783,None,SUBJ_0005
...,...,...,...,...,...
193,D3/SUBJ_0199,D3,TBI_PTE_fm_20_173,None,SUBJ_0199
194,D3/SUBJ_0200,D3,TBI_PTE_fm_20_174,None,SUBJ_0200
195,D3/SUBJ_0201,D3,TBI_PTE_fm_20_180,None,SUBJ_0201
196,D3/SUBJ_0202,D3,TBI_PTE_fm_20_198,None,SUBJ_0202


## Check a validation batch

Validation is deterministic, retains incomplete batches and is not repeated by the training multiplier. ROI, brain and condition targets are independent outputs. Unavailable ROI channels stay at -1 throughout the scan.

In [4]:
import numpy as np

validation_generator = prepared.make_generator(split.validation, training=False, batch_size=1)
images, targets = validation_generator[0]
print("Input:", images.shape)
print("ROI targets:", targets[0].shape)
print("Brain targets:", targets[1].shape)
print("Condition targets (SHAM=0, TBI=1):", targets[2].ravel())
print("Validation batches:", len(validation_generator))

unavailable = np.all(targets[0][0] == -1, axis=(0, 1, 2))
missing_names = np.asarray(config.labels.names)[unavailable]
print("Unannotated channels in this scan:", missing_names.tolist())

del images, targets, validation_generator

Generator configuration:
- Dataset path: D:\Other files\TESI_CNN\MANUSCRIPT_DATASET\normalized
- Dimensions: (80, 80, 80)
- Batch size: 1
- Number of input channels: 1
- Number of classes: 11
- Shuffle: False

Input: (1, 80, 80, 80, 1)
ROI targets: (1, 80, 80, 80, 11)
Brain targets: (1, 80, 80, 80, 1)
Condition targets (SHAM=0, TBI=1): [1.]
Validation batches: 40
Unannotated channels in this scan: ['Contra corpus callosum', 'Contra hippocampus', 'Contra cortex', 'Ipsi corpus callosum', 'Ipsi hippocampus', 'Ipsi cortex']


## Train and evaluate complete test volumes

Training uses augmented centre-biased random crops. The best validation-selected checkpoint is restored before the untouched test cohort is evaluated. Full-volume metrics use physical spacing and retain `unavailable`/empty-mask statuses. All ROI classes are predicted together, regardless of annotation availability; unavailable reference classes are excluded from measurements. Saved ROI and brain masks restore the original input grid and use original annotation IDs after the explicitly configured postprocessing.

The SHAM/TBI head contributes to fitting and is exported with the model. This notebook reports anatomical segmentation metrics; classifier discrimination statistics belong to a separately specified evaluation protocol.

In [5]:
from src.experiments.training import run_training

result = None
if RUN_TRAINING:
    if USE_TBI_PRETRAINING and config.initialization.mode == "scratch":
        raise ValueError("Set TBI_RUN and rerun configuration, or set USE_TBI_PRETRAINING=False for scratch")

    result = run_training(prepared, split, resume_run=RESUME_RUN, callbacks=callbacks)
    print("MouseFull run saved to:", result.run_dir)
else:
    print("Checks complete. Set RUN_TRAINING = True to train or resume MouseFull.")

Checks complete. Set RUN_TRAINING = True to train or resume MouseFull.


## Inspect evaluation and saved artifacts

Plotly uses native VS Code rendering. Do not average unavailable or undefined metric rows as zeros; inspect statuses first. Results contain source dataset, animal, scan, class and physical units. Saved configuration, split, data contracts, selected weights, inference model, history, timings, predictions and metric tables follow the shared output structure. No full training starts until `RUN_TRAINING=True`.

In [6]:
import pandas as pd
from src.experiments.training import history_figure

if result is not None:
    history = history_figure(result)
    history.show()
    display(pd.DataFrame(result.timings))

    if result.evaluations.get("regions"):
        metrics = pd.concat(result.evaluations["regions"], ignore_index=True)
        test_metrics = metrics.loc[metrics["DatasetRole"] == "test"]
        dice = test_metrics.loc[test_metrics["Metric"] == "Dice"]
        columns = ["Subject", "Scan", "Region", "Value", "Status"]
        display(dice[columns])
        print("Full-volume metrics:", result.run_dir / "regions")
        status_columns = ["Region", "Status"]
        status_counts = test_metrics.groupby(status_columns, dropna=False).size()
        display(status_counts.rename("Metric rows"))

## Explore test predictions against ground truth

Run the cell below after training/evaluation, or point `VIEW_RUN` at an existing matching run folder to inspect it without fitting. Saved full-volume `regions.nii.gz` files are loaded directly; no predictions are recomputed. This requires inference and prediction saving to have been enabled.

The table lists every available scan/timepoint and fold. The viewer initially loads up to three rows; set `VIEW_SCAN_INDICES=(0, 4)` to explore other rows. Use the Plotly dropdown to select a scan and reference-oriented plane, then drag the slice slider. Zoom/pan works in each MRI/GT/prediction panel.

Both overlays use the same class colors. The title lists unannotated GT regions: their absence is not evidence of a prediction error. Set `VIEW_REGION` to a foreground **original annotation ID** (for example 13 for the ipsilateral ventricle) to isolate one region. `VIEW_OPACITY` controls the overlay. Previews sample up to 64 slices per plane and 192 pixels per side; set `VIEW_MAX_SLICES=None` for every slice and raise `VIEW_MAX_PIXELS` for more detail. Saved masks are unchanged. The controls use native Plotly rendering.


In [7]:
from src.evaluation.inference import saved_prediction_manifest, prediction_comparison_figure

VIEW_RUN: Path | None = None  # Existing run folder, or use the result above.
VIEW_SCAN_INDICES: tuple[int, ...] | None = None  # None: first three manifest rows.
VIEW_REGION: int | None = None  # Original annotation ID (e.g. 13 = ipsi ventricle); None shows all.
VIEW_OPACITY: float = 0.55
VIEW_MAX_SLICES: int | None = 64
VIEW_MAX_PIXELS: int = 192
VIEW_ROLE: str = 'test'

viewer_role = VIEW_ROLE
viewer_run = VIEW_RUN
current_result = globals().get("result")
if viewer_run is None and current_result is not None:
    viewer_run = current_result.run_dir
if viewer_run is None and RESUME_RUN is not None:
    viewer_run = RESUME_RUN

if viewer_run is None:
    print("Train/evaluate first, or set VIEW_RUN to an existing run folder.")
else:
    prediction_views = saved_prediction_manifest(prepared, viewer_run, role=viewer_role)
    display(prediction_views)

    if prediction_views.empty:
        print("No saved predictions for this role. Check the run, VIEW_ROLE and prediction-saving settings.")
    else:
        prediction_figure = prediction_comparison_figure(
            data=prepared, predictions=prediction_views,
            scan_indices=VIEW_SCAN_INDICES, region=VIEW_REGION,
            opacity=VIEW_OPACITY, max_slices=VIEW_MAX_SLICES,
            max_pixels=VIEW_MAX_PIXELS,
        )
        prediction_figure.show(renderer="vscode")


,Subject,Scan,Group,Fold,DatasetRole,PredictionPath
ViewIndex,,,,,,


No saved predictions for this role. Check the run, VIEW_ROLE and prediction-saving settings.


## Evaluate the best checkpoint after stopping training

Interrupt training, then run this cell independently of `result`. Set `CHECKPOINT_RUN` to the existing **run folder**, select `CHECKPOINT_FOLD`, and enable `RUN_CHECKPOINT_EVALUATION`. After a kernel restart, only the repository setup cell and this cell are needed: the saved experiment supplies architecture, preprocessing, label policy and animal selection.

The default cohort matches this notebook. Fixed-split runs use test animals, CV uses outer holdouts, and historical rat CV uses validation animals. Every selected animal includes all timepoints. A best checkpoint must already exist; an unfinished epoch is not evaluated. The committed selected-weight copy is used when recovery state is available.

This performs full-volume inference only, saves metrics/predictions to a fresh folder outside the training run, and opens the MRI/GT/prediction viewer. Training weights, optimizer, history and completion flags are left intact. Inference can take time; leave the switch False to skip it. Use notebook 09 to reopen an evaluation without recomputing predictions.

The common RARE reference is explicitly used for checkpoint evaluation, including older checkpoints. A model trained with a different orientation receives a changed input convention; those scores should be interpreted as evaluation with an orientation override. Training recovery files remain unchanged.


The independent cell exposes `checkpoint_postprocessing` and always restores original annotation IDs, including when the saved run predates these settings. It writes `prediction_labels.json`, so the viewer uses the exact exported IDs. `CHECKPOINT_REGION` selects an original annotation ID. Changing evaluation settings does not change training targets or weights.


In [ ]:
from src.experiments.config import PostprocessingConfig
from src.evaluation.inference import (
    best_checkpoint_settings, evaluate_saved_model,
    saved_prediction_manifest, prediction_comparison_figure,
)

RUN_CHECKPOINT_EVALUATION: bool = False
CHECKPOINT_RUN: Path | None = (
    REPO_ROOT / "results" / "mouse_full_3drnet"
    / "RUN_20261012_220334_565441"
)  # results/<model>/<run>; required after interruption/restart.
CHECKPOINT_FOLD: int = 0
CHECKPOINT_OUTPUT: Path | None = None  # None creates a fresh evaluation folder.
CHECKPOINT_SCAN_INDICES: tuple[int, ...] | None = None
CHECKPOINT_REGION: int | None = None
CHECKPOINT_MAX_SLICES: int | None = 64
CHECKPOINT_ROLE: str = 'test'

checkpoint_run = CHECKPOINT_RUN
current_result = globals().get("result")
if checkpoint_run is None and current_result is not None:
    checkpoint_run = current_result.run_dir
if checkpoint_run is None:
    checkpoint_run = globals().get("RESUME_RUN")

checkpoint_postprocessing = PostprocessingConfig(
    clean_brain_mask=True,
    fill_small_holes=True, holes_max_area=20000,
    remove_small_objects=True, object_min_area=30000,
    split_ventricles=False, use_centroids=False,
    default_hemisphere="right", lesion_voxels_threshold=10,
)

checkpoint_evaluation = None
if RUN_CHECKPOINT_EVALUATION:
    if checkpoint_run is None:
        raise ValueError("Set CHECKPOINT_RUN to the stopped run folder first")
    checkpoint_settings = best_checkpoint_settings(
        run_dir=checkpoint_run, fold=CHECKPOINT_FOLD, role=CHECKPOINT_ROLE,
        output_dir=CHECKPOINT_OUTPUT, reference=REFERENCE,
        original_label_ids=True, postprocessing=checkpoint_postprocessing,
    )
    print("Selected weights:", checkpoint_settings.model_file)
    checkpoint_evaluation = evaluate_saved_model(checkpoint_settings)
    print("Evaluation saved to:", checkpoint_evaluation.output_dir)
    checkpoint_metrics = checkpoint_evaluation.metrics
    checkpoint_dice = checkpoint_metrics.loc[checkpoint_metrics["Metric"] == "Dice"]
    display(checkpoint_dice)
    checkpoint_views = saved_prediction_manifest(
        data=checkpoint_evaluation.data, run_dir=checkpoint_evaluation.output_dir,
        role=CHECKPOINT_ROLE,
    )
    display(checkpoint_views)
    checkpoint_figure = prediction_comparison_figure(
        data=checkpoint_evaluation.data, predictions=checkpoint_views,
        scan_indices=CHECKPOINT_SCAN_INDICES, region=CHECKPOINT_REGION,
        max_slices=CHECKPOINT_MAX_SLICES,
    )
    checkpoint_figure.show(renderer="vscode")
else:
    print("Set CHECKPOINT_RUN and RUN_CHECKPOINT_EVALUATION=True to evaluate saved best weights.")


Set CHECKPOINT_RUN and RUN_CHECKPOINT_EVALUATION=True to evaluate saved best weights.
